# ShotShield v2 — export for the browser demo

Converts the released model ([v2.0](https://github.com/nishttw/ShotShield/releases/tag/v2.0)) to **ONNX** for
`onnxruntime-web`, and packages reference data used to check the JavaScript front-end against librosa.
Nothing is retrained: the model is downloaded from the release and its SHA-256 is checked.

**Before you run this notebook on Kaggle**
1. **Settings → Internet → On** (it downloads the model and `features.py` from GitHub).
2. **Add Input** → **UrbanSound8K** (the same dataset as training; used for example clips from test fold 10).
3. Accelerator: **None** — a CPU is enough.
4. **Run All**, then download **`shotshield_v2_web.zip`** from Output → `/kaggle/working` (the zip, not single files).

The zip contains:
| File | Used for |
|---|---|
| `shotshield_v2.onnx` | the model for the browser (`docs/model/`) |
| `model_info.json` | the release's info + ONNX checksum and conversion check |
| `examples/*.wav`, `examples/examples.json` | "try an example" clips from test fold 10 (never seen in training) |
| `reference.npz` | waveforms, librosa features and Keras probabilities for `tools/verify_features.py` and `tools/verify_onnx.py` |

In [ ]:
# Conversion tools (not preinstalled on Kaggle). tf2onnx is installed without its dependencies:
# it pins an old protobuf that would break TensorFlow.
!pip install -q onnx onnxruntime onnxscript
!pip install -q --no-deps tf2onnx

In [ ]:
import os, sys, json, glob, hashlib, shutil, subprocess, time, urllib.request, datetime

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from joblib import Parallel, delayed

import tensorflow as tf
import keras
import onnx
import onnxruntime as ort

WORK = '/kaggle/working'
OUT = f'{WORK}/shotshield_web'
TMP = '/tmp/shotshield_export'          # /tmp is not part of the notebook output
shutil.rmtree(OUT, ignore_errors=True)
for d in (OUT, f'{OUT}/examples', TMP):
    os.makedirs(d, exist_ok=True)

REPO_RAW = 'https://raw.githubusercontent.com/nishttw/ShotShield/main'
RELEASE = 'https://github.com/nishttw/ShotShield/releases/download/v2.0'


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def fetch(url, path):
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    return path


# The model from Release v2.0 (or a copy attached under /kaggle/input)
attached = glob.glob('/kaggle/input/**/shotshield_v2.keras', recursive=True)
MODEL_PATH = attached[0] if attached else fetch(f'{RELEASE}/shotshield_v2.keras', f'{TMP}/shotshield_v2.keras')
info = json.load(open(fetch(f'{RELEASE}/model_info.json', f'{TMP}/model_info.json')))
assert sha256(MODEL_PATH) == info['model_sha256'], 'model checksum does not match model_info.json'
print('model:', MODEL_PATH, '— checksum OK')

# The exact feature code from the repository, so the references come from the same code as training
fetch(f'{REPO_RAW}/src/shotshield/features.py', f'{TMP}/shotshield_features.py')
sys.path.insert(0, TMP)
import shotshield_features as F
CFG = F.FeatureConfig.from_dict(info['features'])
THRESHOLD = info['threshold']

VERSIONS = {'tensorflow': tf.__version__, 'keras': keras.__version__, 'librosa': librosa.__version__,
            'numpy': np.__version__, 'onnx': onnx.__version__, 'onnxruntime': ort.__version__}
print(VERSIONS)
if librosa.__version__ != info['versions']['librosa']:
    print(f"NOTE: librosa {librosa.__version__} here, model trained with {info['versions']['librosa']}")

In [ ]:
model = keras.models.load_model(MODEL_PATH)
model.summary()


def keras_predict(feats):
    return model.predict(feats, batch_size=32, verbose=0).ravel()

## 1. Reference signals
Synthetic signals cover the edge cases (noise, tones, sharp clicks, loop padding, clips longer than one window,
near-silence). Real clips come from UrbanSound8K **test fold 10**, which the final model never saw in training.

In [ ]:
rng = np.random.default_rng(2026)
N, SR = CFG.n_samples, CFG.sample_rate
t = np.arange(N) / SR


def burst(n, decay):
    return rng.standard_normal(n) * np.exp(-np.arange(n) / (decay * SR))


shots = np.zeros(N)
for start in (0.4, 1.6, 1.75, 3.1):
    i = int(start * SR)
    shots[i:i + SR // 2] += 0.9 * burst(SR // 2, 0.03)
noise = 0.01 * rng.standard_normal(3 * N)

synthetic = {
    'synth_white_noise': 0.3 * rng.standard_normal(N),
    'synth_chirp': 0.5 * np.sin(2 * np.pi * 50 * 4 / np.log(200) * (200 ** (t / 4) - 1)),
    'synth_impulses_in_noise': shots + noise[:N],
    'synth_short_clip_0.6s': burst(int(0.6 * SR), 0.1),
    'synth_long_7.5s': np.concatenate([noise[:N], shots + noise[N:2 * N]])[: int(7.5 * SR)],
    'synth_quiet_noise': 1e-4 * rng.standard_normal(N),
}
synthetic = {k: v.astype(np.float32) for k, v in synthetic.items()}
print({k: f'{len(v) / SR:.2f} s' for k, v in synthetic.items()})

In [ ]:
# Find UrbanSound8K (folder layout differs between Kaggle versions)
csv_hits, fold1_hits = [], []
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    csv_hits += [os.path.join(root, f) for f in files if f.lower() == 'urbansound8k.csv']
    fold1_hits += [os.path.join(root, d) for d in dirs if d.lower() == 'fold1']
HAVE_US8K = bool(csv_hits and fold1_hits)
examples = []

if not HAVE_US8K:
    print('UrbanSound8K not attached — continuing without example clips (Add Input → UrbanSound8K to include them).')
else:
    meta = pd.read_csv(sorted(csv_hits)[0])
    audio_root = os.path.dirname(sorted(fold1_hits)[0])
    meta['path'] = [os.path.join(audio_root, f'fold{f}', n) for f, n in zip(meta['fold'], meta['slice_file_name'])]
    meta['duration'] = meta['end'] - meta['start']
    fold10 = meta[meta['fold'] == 10].reset_index(drop=True)

    # Score all of test fold 10 the way the notebook evaluated it (one loop-padded 4 s window per clip)
    def first_window(path):
        return F.log_mel(F.loop_pad(F.load_audio(path, CFG), CFG.n_samples), CFG)

    X10 = np.stack(Parallel(n_jobs=-1)(delayed(first_window)(p) for p in fold10['path']))[..., None]
    fold10['prob'] = keras_predict(X10)
    gun = fold10['class'] == 'gun_shot'
    print(f"fold 10: {(fold10.prob[gun] >= THRESHOLD).sum()}/{gun.sum()} gunshots caught, "
          f"{(fold10.prob[~gun] >= THRESHOLD).sum()}/{(~gun).sum()} false alarms at threshold {THRESHOLD:.3f}")

    # Example clips: picked at random (fixed seed), not for being easy, plus the non-gunshot the model
    # scores highest — so the demo also shows where the model struggles.
    picks = [fold10[gun].sample(2, random_state=7)]
    for cls in ['dog_bark', 'siren', 'drilling', 'jackhammer', 'air_conditioner', 'car_horn']:
        picks.append(fold10[fold10['class'] == cls].sample(1, random_state=7))
    hardest = fold10[~gun].nlargest(1, 'prob')
    picks.append(hardest.assign(note='the non-gunshot clip in test fold 10 the model scores highest'))
    chosen = pd.concat(picks).drop_duplicates('slice_file_name')

    for _, r in chosen.iterrows():
        name = f"{r['class']}_{r['fsID']}_{os.path.splitext(r['slice_file_name'])[0].split('-', 1)[1]}"
        y = F.load_audio(r['path'], CFG)
        wav = f'{OUT}/examples/{name}.wav'
        sf.write(wav, np.clip(np.round(y * 32767), -32768, 32767).astype(np.int16), SR, subtype='PCM_16')
        y16, _ = sf.read(wav, dtype='float32')   # what the browser decodes: int16 / 32768
        examples.append({'name': name, 'file': f'examples/{name}.wav', 'class': r['class'],
                         'is_gunshot': bool(r['class'] == 'gun_shot'), 'duration': round(len(y16) / SR, 3),
                         'fold': 10, 'slice_file_name': r['slice_file_name'], 'fsID': int(r['fsID']),
                         'source': f"https://freesound.org/s/{int(r['fsID'])}/",
                         'note': r.get('note') if isinstance(r.get('note'), str) else None,
                         '_wave': y16})
    display(chosen[['slice_file_name', 'class', 'duration', 'prob']])

## 2. Features and Keras predictions for every reference signal
Exactly `features.windows_to_features` (4 s windows, 1 s hop, loop padding) and the Keras model.

In [ ]:
waves = dict(synthetic)
waves.update({e['name']: e.pop('_wave') for e in examples})

arrays, feats_all, probs_all = {}, [], []
for name, y in waves.items():
    feats, starts = F.windows_to_features(y, CFG)
    probs = keras_predict(feats)
    arrays[f'wave/{name}'] = y.astype(np.float32)
    arrays[f'feat/{name}'] = feats.astype(np.float32)
    arrays[f'starts/{name}'] = np.asarray(starts, np.float64)
    arrays[f'prob/{name}'] = probs.astype(np.float32)
    feats_all.append(feats); probs_all.append(probs)

for e in examples:
    p = arrays[f"prob/{e['name']}"]
    e['keras_window_probs'] = [round(float(v), 6) for v in p]
    e['keras_max_prob'] = round(float(p.max()), 6)
    e['keras_detected'] = bool(p.max() >= THRESHOLD)

X_ref = np.concatenate(feats_all).astype(np.float32)
p_keras = np.concatenate(probs_all)
ref_info = {'librosa': librosa.__version__, 'keras': keras.__version__, 'tensorflow': tf.__version__,
            'threshold': THRESHOLD, 'created': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')}
np.savez_compressed(f'{OUT}/reference.npz', info=json.dumps(ref_info), **arrays)
print(f'{len(waves)} signals, {len(X_ref)} windows → reference.npz ({os.path.getsize(OUT + "/reference.npz") / 1e6:.1f} MB)')

## 3. Convert to ONNX
Three converters are tried, each in its own process (a failure can't leave TensorFlow in a bad state).
Every result is run with onnxruntime on all reference windows and compared with Keras; the first one that
matches is kept.

In [ ]:
CONVERT = r'''
import os, sys
strategy, model_path, out_path = sys.argv[1:4]
if strategy == 'keras_torch':
    os.environ['KERAS_BACKEND'] = 'torch'
import keras
model = keras.models.load_model(model_path)
import numpy as np
model(np.zeros((1,) + tuple(model.inputs[0].shape[1:]), 'float32'))   # export needs a model that has been called once
if strategy == 'tf2onnx':
    import tensorflow as tf, tf2onnx
    spec = (tf.TensorSpec((None,) + tuple(model.inputs[0].shape[1:]), tf.float32, name='log_mel'),)
    fn = tf.function(lambda x: model(x, training=False), input_signature=spec)
    tf2onnx.convert.from_function(fn, input_signature=spec, opset=17, output_path=out_path)
else:
    model.export(out_path, format='onnx')
print('converted with', strategy, 'on backend', keras.backend.backend())
'''

attempts, chosen = [], None
for strategy in ['keras_tf', 'tf2onnx', 'keras_torch']:
    path = f'{TMP}/model_{strategy}.onnx'
    t0 = time.time()
    r = subprocess.run([sys.executable, '-c', CONVERT, strategy, MODEL_PATH, path],
                       capture_output=True, text=True, timeout=1200)
    row = {'strategy': strategy, 'seconds': round(time.time() - t0)}
    if r.returncode != 0 or not os.path.exists(path):
        row['error'] = (r.stderr or r.stdout).strip().splitlines()[-1:] or ['unknown error']
        print(f'{strategy}: FAILED — {row["error"][0][:300]}')
        attempts.append(row); continue
    try:
        m = onnx.load(path)               # also pulls in any external weight file
        onnx.checker.check_model(m)
        sess = ort.InferenceSession(m.SerializeToString(), providers=['CPUExecutionProvider'])
        inp = sess.get_inputs()[0]
        # batch size 1, the way the browser runs it
        p_onnx = np.concatenate([sess.run(None, {inp.name: X_ref[i:i + 1]})[0].ravel() for i in range(len(X_ref))])
        err = float(np.abs(p_onnx - p_keras).max())
        same = int(((p_onnx >= THRESHOLD) == (p_keras >= THRESHOLD)).sum())
        row.update(max_abs_diff=err, same_decision=f'{same}/{len(X_ref)}', input=inp.name, input_shape=str(inp.shape),
                   opset=max(o.version for o in m.opset_import if o.domain in ('', 'ai.onnx')),
                   ops=sorted({n.op_type for n in m.graph.node}))
        print(f"{strategy}: max |ONNX − Keras| = {err:.2e}, same decision {row['same_decision']}, "
              f"input {inp.name} {inp.shape}, opset {row['opset']}")
        if chosen is None and err < 1e-4 and same == len(X_ref):
            chosen = (strategy, m, row)
    except Exception as e:
        row['error'] = [repr(e)]
        print(f'{strategy}: converted but failed the check — {e!r}')
    attempts.append(row)

assert chosen is not None, 'No converter produced a matching ONNX model — send the output above.'
strategy, onnx_model, check = chosen
print(f'\nUsing {strategy}. Ops: {", ".join(check["ops"])}')

## 4. Package

In [ ]:
ONNX_PATH = f'{OUT}/shotshield_v2.onnx'
onnx.save(onnx_model, ONNX_PATH)   # single file, weights inline

web_info = dict(info)
web_info['web'] = {
    'onnx_file': 'shotshield_v2.onnx', 'onnx_sha256': sha256(ONNX_PATH), 'converter': strategy,
    'input_name': check['input'], 'input_shape': check['input_shape'], 'opset': check['opset'],
    'max_abs_diff_vs_keras': check['max_abs_diff'], 'checked_windows': int(len(X_ref)),
    'attempts': attempts, 'versions': VERSIONS,
    'created': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
}
json.dump(web_info, open(f'{OUT}/model_info.json', 'w'), indent=2)
json.dump({'dataset': 'UrbanSound8K (J. Salamon, C. Jacoby, J. P. Bello, 2014), CC BY-NC 3.0; '
                      'clips originally from freesound.org',
           'threshold': THRESHOLD, 'examples': examples}, open(f'{OUT}/examples/examples.json', 'w'), indent=2)

ZIP = shutil.make_archive(f'{WORK}/shotshield_v2_web', 'zip', OUT)
for root, _, files in os.walk(OUT):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f'{os.path.relpath(p, OUT):60s} {os.path.getsize(p) / 1024:8.1f} KB')
print(f'\n{ZIP}: {os.path.getsize(ZIP) / 1e6:.2f} MB, sha256 {sha256(ZIP)}')
if examples:
    display(pd.DataFrame(examples)[['name', 'class', 'duration', 'keras_max_prob', 'keras_detected']])